In [ ]:
pip install qiskit

In [ ]:
pip install qiskit_nature

In [ ]:
pip install qiskit-aer

In [ ]:
pip install pyscf

In [ ]:
pip install qiskit_ibm_runtime

In [ ]:
pip install pylatexenc

## Definiendo problemas moleculares

In [ ]:
from qiskit_nature.second_q.drivers import PySCFDriver

driver = PySCFDriver(atom="H 0 0 0; H 0 0 0.735")
problem = driver.run()

hamiltonian = problem.hamiltonian.second_q_op()

print(hamiltonian)

In [ ]:
from qiskit_nature.second_q.mappers import ParityMapper

mapper = ParityMapper(num_particles=problem.num_particles)

qubit_op = mapper.map(hamiltonian)
aux_ops = {}
aux_ops.update(mapper.map(problem.properties.particle_number.second_q_ops()))
aux_ops.update(mapper.map(problem.properties.angular_momentum.second_q_ops()))

print(qubit_op)

In [ ]:
from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver
solver = NumPyMinimumEigensolver()
result = solver.compute_minimum_eigenvalue(qubit_op)
print(result)

## Resolviendo el problema con VQE

In [ ]:
from qiskit.circuit.library import efficient_su2

ansatz = efficient_su2(
    qubit_op.num_qubits, su2_gates=["h", "rz", "y"], entanglement="circular", reps=1
)
num_params = ansatz.num_parameters
print("This circuit has", num_params, "parameters")

ansatz.decompose().draw("mpl", style="iqp")

In [ ]:
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = AerSimulator(method="statevector")


In [ ]:
def cost_func(params, ansatz, qubit_op, estimator):
    """Return estimate of energy from estimator

    Parameters:
        params (ndarray): Array of ansatz parameters
        ansatz (QuantumCircuit): Parameterized ansatz circuit
        hamiltonian (SparsePauliOp): Operator representation of Hamiltonian
        estimator (EstimatorV2): Estimator primitive instance
        cost_history_dict: Dictionary for storing intermediate results

    Returns:
        float: Energy estimate
    """
    pub = (ansatz, [qubit_op], params)
    result = estimator.run(pubs=[pub]).result()
    energy = result[0].data.evs[0]

    cost_history_dict["iters"] += 1
    cost_history_dict["prev_vector"] = params
    cost_history_dict["cost_history"].append(energy)
    print(f"Iters. done: {cost_history_dict['iters']} [Current cost: {energy}]")

    return energy


cost_history_dict = {
    "prev_vector": None,
    "iters": 0,
    "cost_history": [],
}

In [ ]:
from qiskit_ibm_runtime import Session
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from scipy.optimize import minimize
import numpy as np
from qiskit.primitives import StatevectorEstimator


estimator = StatevectorEstimator()

num_params = ansatz.num_parameters
x0 = [0.1] * num_params


estimator = StatevectorEstimator()

res = minimize(
        cost_func,
        x0,
        args=(ansatz, qubit_op, estimator),
        method="cobyla",
        options={"maxiter": 500},
    )

In [ ]:
print(res)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
x = np.linspace(0, 10, res.nfev)

# Define the constant function
constant = -1.857275030202381
y_constant = np.full_like(x, constant)
ax.plot(
    range(cost_history_dict["iters"]), cost_history_dict["cost_history"], label="VQE"
)
ax.set_xlabel("Iterations")
ax.set_ylabel("Cost")
ax.plot(y_constant, label="Target")
plt.legend()
plt.draw()